In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT=Path.cwd().parent
df=pd.read_csv(ROOT/'data'/'raw'/'Social_Media_Advertising.csv')
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.head())


#Check unique values in columns
for col in ['Channel_Used','Campaign_Goal','Target_Audience','Customer_Segment']:
        print(f'{col}: {df[col].unique()}')

#Standardize column names
df.columns=df.columns.str.strip().str.replace(' ','_').str.lower()


(300000, 16)
Campaign_ID           int64
Target_Audience      object
Campaign_Goal        object
Duration             object
Channel_Used         object
Conversion_Rate     float64
Acquisition_Cost     object
ROI                 float64
Location             object
Language             object
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment     object
Date                 object
Company              object
dtype: object
Campaign_ID         0
Target_Audience     0
Campaign_Goal       0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Location            0
Language            0
Clicks              0
Impressions         0
Engagement_Score    0
Customer_Segment    0
Date                0
Company             0
dtype: int64
   Campaign_ID Target_Audience     Campaign_Goal Duration Channel_Used  \
0       529013       Men 35-44    Product Launch  15 Days    Instagram   
1       2753

In [13]:
# Check what acquisition_cost actually looks like
print(df['acquisition_cost'].head(10))
print(df['acquisition_cost'].dtype)

# Remove $ signs and commas, convert to float
df['acquisition_cost'] = (
    df['acquisition_cost']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

# Verify it worked
print(df['acquisition_cost'].dtype)   # must say float64
print(df['acquisition_cost'].head())
#Parse dates
df['date'] = pd.to_datetime(df['date'])
df['month']=df['date'].dt.to_period('M').astype(str)
df['day_of_week']=df['date'].dt.day_name()

#Calculate marketing metrics
#CTR= Clicks/Impressions*100
df['ctr']=(df['clicks']/df['impressions']*100).round(4)

#CPC=Acquisition Cost/Clicks
df['cpc']=(df['acquisition_cost']/df['clicks'].replace(0,np.nan)).round(4)

#CPM=Acquisition Cost/Impressions*1000
df['cpm']=(df['acquisition_cost']/df['impressions']*1000).round(4)

#Revenue estimate=Acqisition Cost *(1+ROI)
df['estimated_revenue']=(df['acquisition_cost']*(1+df['roi'])).round(2)

#Profit=Revenue-Cost
df['profit']=(df['estimated_revenue']-df['acquisition_cost']).round(2)

#Conversions estimate=Clicks*Conversion Rate
df['conversions']=(df['clicks']*df['conversion_rate']).round(0).astype(int)

df['roas'] = (df['estimated_revenue'] / df['acquisition_cost']).round(3)

print(df[['ctr','cpc','cpm','roi','conversions']].describe())

#Saved cleaned data
df.to_csv(ROOT/'data'/'processed'/'ads_cleaned.csv', index=False)
print('Saved ads_cleaned.csv')

0    500.0
1    500.0
2    500.0
3    500.0
4    500.0
5    500.0
6    500.0
7    500.0
8    500.0
9    500.0
Name: acquisition_cost, dtype: float64
float64
float64
0    500.0
1    500.0
2    500.0
3    500.0
4    500.0
Name: acquisition_cost, dtype: float64
                 ctr            cpc            cpm            roi  \
count  300000.000000  300000.000000  300000.000000  300000.000000   
mean       31.415603       0.455768     141.221250       3.177691   
std         2.465685       0.129101      31.652637       2.461200   
min        15.118700       0.239800      64.977300       0.000000   
25%        30.169775       0.380000     125.692800       0.930000   
50%        32.537300       0.387500     126.556700       2.670000   
75%        33.126600       0.572100     130.208300       5.330000   
max        33.333300       1.706500     258.131100       8.000000   

         conversions  
count  300000.000000  
mean     1453.380080  
std      1234.338241  
min         3.000000  
25% 